## Surrogate vs. Thermochemical library
### **Space-time analysis**
Import libraries

In [ ]:
import os
import sys

In [ ]:
sys.path.append(os.path.abspath('./../functions'))

import error as err
import reading as read
import plotting as plt

%load_ext kernel_manager

Define inputs

In [ ]:
models = ['surrogate', 'thermolib']
scheme = 'strang'
nb_nodes = 1401
nb_iters = 2200
qoi = {
    'names': ['Y_O', 'Y_O2', 'Th', 'Tv'],
    'labels': {
        'Y_O2': (r"$Y_{\text{O}_2}$",),
        'Y_O': (r"$Y_{\text{O}}$",),
        'Th': (r"$T$", "[K]"),
        'Tv': (r"$T_{\text{v}}$", "[K]")
    },
    'format': {
        'Y_O2': '%.2f',
        'Y_O': '%.2f',
        'Th': None,
        'Tv': None
    },
    'cticks': {
        'Y_O2': True,
        'Y_O': True,
        'Th': False,
        'Tv': False
    }
}
path_to_case = "./../../"

In [ ]:
skip_flags = {
    'diagram': False,
    'error': False,
    'profiles': False
}

Load data

In [ ]:
data = read.load_dataset(path_to_case, nb_nodes, nb_iters)

Global variables

In [ ]:
t = data['thermolib']['t']
x = data['thermolib']['x']

In [ ]:
path_to_saving = os.path.abspath("./../output/space_time/")
os.makedirs(path_to_saving, exist_ok=True)

### Space-time diagram

Single QoI

In [ ]:
%%skip $skip_flags['diagram']

for k in qoi['names']:
    for model in models:
        plt.time_space_2d(
            t,
            x,
            y=data[model][k],
            tlim=[0., 1.e-5],
            xlim=[0.004, 0.009],
            clbl=' '.join(qoi['labels'][k]),
            cfmt=qoi['format'][k],
            compute_cticks=qoi['cticks'][k],
            interp='bilinear',
            figname=f'{model}.{k}',
            path=path_to_saving
        )

Relative error

In [ ]:
%%skip $skip_flags['error']

shock_pos = {model: err.capture_shock(data[model]['u']) for model in models}
e = {
    k: err.compute_err_2d(
        y={model: data[model][k] for model in models},
        x=x,
        shift=shock_pos,
        lin_flt=True if k == 'Tv' else False,
        lin_flt_idx=(500, 900)
    ) for k in qoi['names']
}

In [ ]:
%%skip $skip_flags['error']

for k in qoi['names']:
    plt.time_space_2d(
        t,
        x,
        y=e[k],
        tlim=[0., 1.e-5],
        xlim=[0.004, 0.009],
        clbl='Rel. error [\%]',
        cfmt='%.1f',
        compute_cticks=False,
        interp='bilinear',
        figname=f'error.{k}',
        path=path_to_saving
    )